# 02 — Padronização e correção geométrica

**Objetivo:** corrigir a projeção quebrada dos setores de 2000, unir urbano+rural num único layer por cidade,
filtrar por município, harmonizar tipos de coluna entre os três anos e preparar as 15 camadas (5 cidades × 3
anos) que servem de entrada para a compatibilização areal do notebook 03.

**Entradas:**
- `input/<Cidade>/<ano>/*.shp` (20 shapefiles)

**Saídas:**
- `output/02_padronizado/<slug_cidade>_<ano>.gpkg` (15 arquivos, 1 layer cada)

**Depende de:** `01_diagnostico_dados.ipynb`

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "utils.py").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

import geopandas as gpd
import pandas as pd
from shapely.validation import make_valid

from utils import (
    CIDADES, ANOS, OUTPUT_DIR,
    caminho_shapefile, ENCODING_POR_ANO, CRS_WGS84,
    CRS_2000_URBANO, CRS_METRICO, CANDIDATOS_DATUM_2000,
)

OUT_DIR = OUTPUT_DIR / "02_padronizado"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


## 2. Correção do CRS de 2000-urbano

- O `.prj` declara zona UTM Norte (WGS84), mas as coordenadas (northing ~6,8 a 9,7 milhões) são do hemisfério
  Sul.
- Correção: **reatribuição de CRS** (não reprojeção) — sobrescreve o CRS declarado pela zona Sul equivalente
  (mesmo datum WGS84, mesmo fuso, `false_northing` correto), sem alterar nenhuma coordenada.

In [ ]:
def carrega_2000_urbano_corrigido(cidade):
    caminho = caminho_shapefile(cidade, 2000, "urbano")
    gdf = gpd.read_file(caminho, encoding=ENCODING_POR_ANO[2000])
    crs_correto = CRS_2000_URBANO[cidade]
    gdf = gdf.set_crs(crs_correto, allow_override=True)
    gdf["situacao_fonte"] = "URBANO"
    return gdf


exemplo = carrega_2000_urbano_corrigido("Anápolis")
print("CRS corrigido:", exemplo.crs)
exemplo.total_bounds


## 3. Diagnóstico de datum para 2000-rural (sem `.prj`)

- Os shapefiles rurais de 2000 não têm CRS definido; as coordenadas estão em graus decimais, na região
  aproximada correta (sistema geográfico, não UTM).
- Testa três hipóteses de datum — WGS84, SIRGAS 2000 e SAD69 — atribuindo cada uma, reprojetando para o CRS
  métrico da cidade, e comparando a área de interseção com o município da malha de 2022 (IoU: intersection
  over union). A hipótese vencedora é a de maior sobreposição.
- Premissa: usa apenas as feições com `V1` não nulo e sem `GEOCODIGO` duplicado (ver seção 3.1) — os polígonos
  administrativos residuais do arquivo bruto distorceriam a comparação de área.

In [ ]:
def uniao_valida(gdf):
    '''União de todas as geometrias de um GeoDataFrame, corrigindo invalidades antes (evita GEOSException).'''
    geoms = gdf.geometry.apply(lambda g: g if g is not None and g.is_valid else make_valid(g))
    return geoms.union_all()


def carrega_2022_municipio(cidade):
    gdf = gpd.read_file(caminho_shapefile(cidade, 2022), encoding=ENCODING_POR_ANO[2022])
    return gdf.to_crs(CRS_METRICO[cidade])


def iou_datum(gdf_teste_wgs, gdf_2022_metrico, crs_candidato, crs_metrico):
    gdf = gdf_teste_wgs.set_crs(crs_candidato, allow_override=True).to_crs(crs_metrico)
    geom_teste = uniao_valida(gdf)
    geom_2022 = uniao_valida(gdf_2022_metrico)
    inter = geom_teste.intersection(geom_2022).area
    uniao = geom_teste.union(geom_2022).area
    return inter / uniao if uniao > 0 else 0.0


resultados_datum = []
melhor_datum_rural = {}

for cidade in CIDADES:
    caminho_rural = caminho_shapefile(cidade, 2000, "rural")
    gdf_rural_bruto = gpd.read_file(caminho_rural, encoding=ENCODING_POR_ANO[2000])
    gdf_rural_bruto = gdf_rural_bruto[gdf_rural_bruto["V1"].notna()].drop_duplicates(subset=["GEOCODIGO"])
    gdf_2022_metrico = carrega_2022_municipio(cidade)

    scores = {}
    for nome_datum, epsg in CANDIDATOS_DATUM_2000.items():
        score = iou_datum(gdf_rural_bruto, gdf_2022_metrico, epsg, CRS_METRICO[cidade])
        scores[nome_datum] = score
        resultados_datum.append({"cidade": cidade, "datum_candidato": nome_datum, "epsg": epsg, "iou": score})

    melhor = max(scores, key=scores.get)
    melhor_datum_rural[cidade] = melhor

df_datum = pd.DataFrame(resultados_datum)
df_datum.pivot(index="cidade", columns="datum_candidato", values="iou")


In [ ]:
print("Datum escolhido por cidade para o 2000-rural (maior IoU com o município 2022):")
for cidade, datum in melhor_datum_rural.items():
    print(f"  {cidade}: {datum} ({CANDIDATOS_DATUM_2000[datum]})")


- Repete a conferência para o **2000-urbano já com a correção de hemisfério aplicada**, usando **cobertura**
  em vez de IoU: a mancha urbana é fração pequena da área do município, então IoU contra o município inteiro
  seria baixo mesmo com CRS correto.
- Métrica: fração da área urbana de 2000 que cai dentro do polígono do município em 2022
  (`interseção / área_urbana`). Premissa: correção certa → fração próxima de 1 (perdendo só o que se explica
  por mudança de limite urbano/expansão entre os censos).

In [ ]:
resultados_urbano = []
for cidade in CIDADES:
    gdf_urbano_corrigido = carrega_2000_urbano_corrigido(cidade)  # já no datum/zona corretos (WGS84 sul)
    gdf_2022_metrico = carrega_2022_municipio(cidade)
    gdf_metrico = gdf_urbano_corrigido.to_crs(CRS_METRICO[cidade])
    geom_urbano = uniao_valida(gdf_metrico)
    geom_2022 = uniao_valida(gdf_2022_metrico)
    cobertura = geom_urbano.intersection(geom_2022).area / geom_urbano.area
    resultados_urbano.append({"cidade": cidade, "cobertura_urbano_dentro_municipio_2022": cobertura})

pd.DataFrame(resultados_urbano)


Cobertura próxima de 1 confirma que a correção de hemisfério foi suficiente — a mancha urbana de 2000 cai, quase
inteiramente, dentro do território municipal de 2022. Não é necessário testar outros datums para o 2000-urbano.

## 3.1. Qualidade do 2000-rural: feições sem população e duplicatas

- Antes de unir urbano+rural, checa a integridade do rural-2000. Nas cinco cidades, uma fração das feições tem
  **`V1` nulo** — não são setores censitários com dado de população, e sim polígonos administrativos maiores
  (prováveis contornos de distrito) cujo `GEOCODIGO` tem formato diferente (com hífen, ex.
  `520110810000001-0001`), que se sobrepõem geometricamente aos setores reais.
- Há também **`GEOCODIGO` duplicado** em todas as cidades (feições idênticas repetidas).
- Sem filtrar isso, a interpolação areal do notebook 03 contaria área (e, em alguns casos, população) mais de
  uma vez.

In [ ]:
relatorio_qualidade_rural = []
for cidade in CIDADES:
    rural_bruto = gpd.read_file(caminho_shapefile(cidade, 2000, "rural"), encoding=ENCODING_POR_ANO[2000])
    n_nulos = int(rural_bruto["V1"].isna().sum())
    n_dup = int(rural_bruto.duplicated(subset=["GEOCODIGO"]).sum())
    area_nulos_km2 = float(rural_bruto.loc[rural_bruto["V1"].isna(), "AREA_TOT_G"].sum())
    relatorio_qualidade_rural.append({
        "cidade": cidade, "n_feicoes": len(rural_bruto),
        "n_sem_populacao_V1_nulo": n_nulos, "n_geocodigo_duplicado": n_dup,
        "area_removida_km2": round(area_nulos_km2, 1),
    })

df_qualidade_rural = pd.DataFrame(relatorio_qualidade_rural)
df_qualidade_rural


## 4. União urbano + rural, filtro municipal e harmonização

- 2000: concatena urbano (CRS corrigido) e rural (datum escolhido pelo teste de IoU), filtra por código
  municipal (resolve o caso de Vitória da Conquista, cujo rural-2000 inclui setores de município vizinho) e
  padroniza nomes/tipos de coluna.
- 2010 e 2022: apenas harmonização de tipos (ex.: V19/V21/V25 vêm como `Real` em 2022 e `Integer64` em 2010) e
  correção de geometrias inválidas.

- Antes de unir urbano+rural (2000) ou de exportar (2010/2022), seleciona apenas as colunas relevantes ao
  algoritmo (`id_setor_origem`, `situacao_fonte`, as variáveis V1-V54 existentes, e a geometria).
- Os shapefiles de origem trazem colunas técnicas com nomes que colidem entre si de forma
  **case-insensitive** (ex.: `Situacao` em 2000-urbano vs. `SITUACAO` em 2000-rural) — o formato
  GeoPackage/SQLite não distingue maiúsculas de minúsculas em nomes de campo, e gravar as duas juntas quebra a
  escrita. Como essas colunas técnicas não fazem parte do dicionário de variáveis, são descartadas.

In [ ]:
VARS_COMUNS = [
    "V1", "V3", "V5", "V7", "V9", "V11", "V13", "V15", "V17", "V19", "V21", "V22",
    "V24", "V24A", "V25", "V26", "V27", "V28", "V30", "V32", "V34", "V36", "V38",
    "V40", "V42", "V44", "V46", "V48", "V50", "V52", "V54",
]


def seleciona_colunas(gdf, chave, novo_nome_chave="id_setor_origem", extras=None):
    '''Mantém apenas a chave de origem (renomeada), as VARS_COMUNS existentes, extras e a geometria.'''
    colunas = [chave] + [c for c in VARS_COMUNS if c in gdf.columns] + (extras or [])
    gdf = gdf[colunas + ["geometry"]].rename(columns={chave: novo_nome_chave})
    return gdf


def monta_2000(cidade):
    urbano = carrega_2000_urbano_corrigido(cidade).to_crs(CRS_WGS84)
    cod_mun = CIDADES[cidade]["cod_municipio"]
    urbano = urbano[urbano["ID_"].astype(str).str.startswith(cod_mun)]
    urbano = seleciona_colunas(urbano, "ID_", extras=["situacao_fonte"])

    rural = gpd.read_file(caminho_shapefile(cidade, 2000, "rural"), encoding=ENCODING_POR_ANO[2000])
    rural = rural[rural["V1"].notna()]  # remove polígonos administrativos sem dado de setor (achado 3.1)
    rural = rural.drop_duplicates(subset=["GEOCODIGO"])  # remove feições duplicadas (achado 3.1)
    rural = rural.set_crs(CANDIDATOS_DATUM_2000[melhor_datum_rural[cidade]], allow_override=True)
    rural["situacao_fonte"] = "RURAL"
    rural = rural.to_crs(CRS_WGS84)
    rural = rural[rural["GEOCODIGO"].astype(str).str.startswith(cod_mun)]
    rural = seleciona_colunas(rural, "GEOCODIGO", extras=["situacao_fonte"])

    gdf = pd.concat([urbano, rural], ignore_index=True)
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=CRS_WGS84)
    return gdf


def harmoniza_tipos(gdf):
    '''Garante float64 em todas as VARS_COMUNS, com NaN real (não `pd.NA`) para colunas ausentes no ano.

    Importante: `pd.NA` sobrevive a um round-trip por GeoPackage como a *string* "<NA>", não como nulo —
    o que quebra qualquer conta aritmética feita depois de reler o arquivo. `float("nan")` grava como nulo
    de verdade em GPKG/SQLite.
    '''
    for col in VARS_COMUNS:
        if col not in gdf.columns:
            gdf[col] = float("nan")
        else:
            gdf[col] = pd.to_numeric(gdf[col], errors="coerce").astype("float64")
    return gdf


def corrige_geometrias(gdf):
    gdf = gdf.copy()
    invalidas_antes = (~gdf.is_valid).sum()
    gdf["geometry"] = gdf["geometry"].apply(lambda g: make_valid(g) if g is not None and not g.is_valid else g)
    invalidas_depois = (~gdf.is_valid).sum()
    return gdf, invalidas_antes, invalidas_depois


In [ ]:
relatorio_geometrias = []

for cidade in CIDADES:
    cod_mun = CIDADES[cidade]["cod_municipio"]
    slug = CIDADES[cidade]["slug"]

    # ---- 2000 ----
    gdf_2000 = monta_2000(cidade)
    gdf_2000 = harmoniza_tipos(gdf_2000)
    gdf_2000, inv_antes, inv_depois = corrige_geometrias(gdf_2000)
    gdf_2000["area_m2"] = gdf_2000.to_crs(CRS_METRICO[cidade]).geometry.area
    relatorio_geometrias.append({"cidade": cidade, "ano": 2000, "n_setores": len(gdf_2000),
                                  "invalidas_antes": inv_antes, "invalidas_depois": inv_depois})
    gdf_2000.to_file(OUT_DIR / f"{slug}_2000.gpkg", driver="GPKG")

    # ---- 2010 ----
    gdf_2010 = gpd.read_file(caminho_shapefile(cidade, 2010), encoding=ENCODING_POR_ANO[2010])
    gdf_2010 = gdf_2010[gdf_2010["CD_GEOCODM"].astype(str).str.startswith(cod_mun)]
    gdf_2010 = seleciona_colunas(gdf_2010, "CD_GEOCODI")
    gdf_2010 = harmoniza_tipos(gdf_2010)
    gdf_2010, inv_antes, inv_depois = corrige_geometrias(gdf_2010)
    gdf_2010["area_m2"] = gdf_2010.to_crs(CRS_METRICO[cidade]).geometry.area
    relatorio_geometrias.append({"cidade": cidade, "ano": 2010, "n_setores": len(gdf_2010),
                                  "invalidas_antes": inv_antes, "invalidas_depois": inv_depois})
    gdf_2010.to_file(OUT_DIR / f"{slug}_2010.gpkg", driver="GPKG")

    # ---- 2022 ----
    gdf_2022 = gpd.read_file(caminho_shapefile(cidade, 2022), encoding=ENCODING_POR_ANO[2022])
    gdf_2022 = gdf_2022[gdf_2022["CD_MUN"].astype(str).str.startswith(cod_mun)]
    gdf_2022 = seleciona_colunas(gdf_2022, "CD_SETOR", extras=["AREA_KM2"])
    gdf_2022 = harmoniza_tipos(gdf_2022)
    gdf_2022, inv_antes, inv_depois = corrige_geometrias(gdf_2022)
    gdf_2022["area_m2"] = gdf_2022.to_crs(CRS_METRICO[cidade]).geometry.area
    relatorio_geometrias.append({"cidade": cidade, "ano": 2022, "n_setores": len(gdf_2022),
                                  "invalidas_antes": inv_antes, "invalidas_depois": inv_depois})
    gdf_2022.to_file(OUT_DIR / f"{slug}_2022.gpkg", driver="GPKG")

    print(f"{cidade}: 2000={len(gdf_2000)}  2010={len(gdf_2010)}  2022={len(gdf_2022)} setores")

df_geom = pd.DataFrame(relatorio_geometrias)
df_geom


## 5. Conferência de área total por cidade/ano

Diferenças grandes entre a área municipal de 2000/2010 e a de 2022 sinalizam problema residual de CRS ou de
filtro municipal.

In [ ]:
linhas_area = []
for cidade in CIDADES:
    slug = CIDADES[cidade]["slug"]
    for ano in ANOS:
        gdf = gpd.read_file(OUT_DIR / f"{slug}_{ano}.gpkg")
        area_km2 = gdf.to_crs(CRS_METRICO[cidade]).geometry.area.sum() / 1e6
        linhas_area.append({"cidade": cidade, "ano": ano, "area_km2": round(area_km2, 2)})

df_area = pd.DataFrame(linhas_area).pivot(index="cidade", columns="ano", values="area_km2")
df_area["dif_2000_vs_2022_%"] = 100 * (df_area[2000] - df_area[2022]) / df_area[2022]
df_area["dif_2010_vs_2022_%"] = 100 * (df_area[2010] - df_area[2022]) / df_area[2022]
df_area


- Diferenças de alguns pontos percentuais são esperadas (mudança de limite municipal e generalização
  cartográfica entre censos); dezenas de por cento indicariam erro de CRS ou de filtro.
- **Santarém: redução territorial real em 2012, não defeito no arquivo de origem.** A malha de 2022 soma
  ~17.900 km², contra ~22.900 km² em 2010; a própria coluna `AREA_KM2` do shapefile de 2022 (calculada pelo
  IBGE) já soma ~17.899 km², confirmando que não é erro de CRS/filtro do nosso lado. `ymin` de 2022 é -3,27°,
  contra -3,56° em 2010: falta a porção sul do território, hoje de outro município.
- Premissa do projeto: a série histórica (2000/2010) é compatibilizada apenas para o território de 2022 — a
  população da área excluída em 2000/2010 não é redistribuída para os setores remanescentes (ver notebook 03,
  seção 4).